In [76]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from settings import x_token, s_token, host, postgre_bd, user, psw
import funcs_yum as fy

In [77]:
posgdb = fy.PostgresDB(host, postgre_bd, user, psw)

# main

## Основная информация с сайта:  
количество аниме на сайте, общее количество жанров, общее количество студий, суммарное число посещений, число оставленных комментариев на сайте

In [78]:
df = pd.read_sql_query(
    '''
    SELECT 
        COUNT(DISTINCT m.anime_id) AS "общее количество аниме",
        COUNT(DISTINCT g.genre_id) AS "общее количество жанров",
        COUNT(DISTINCT s.studio_id) AS "общее количество студий",
        (SELECT SUM(views) FROM anime_main) AS "суммарное количество посещений",
        (SELECT SUM(comments_count) FROM anime_main) AS "количество оставленных комментариев",
        ROUND(AVG(r.y_rating_avg), 2) AS "средний рейтиг всех аниме"
    FROM anime_main AS m
        LEFT JOIN ratings AS r ON m.anime_id = r.anime_id
        LEFT JOIN genres AS g ON m.anime_id = g.anime_id
        LEFT JOIN studios AS s ON m.anime_id = s.anime_id
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\2457532725.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,общее количество аниме,общее количество жанров,общее количество студий,суммарное количество посещений,количество оставленных комментар,средний рейтиг всех аниме
0,10197,104,1155,883997869,536711,6.61


## топ 20 аниме по рейтингу

In [79]:
df = pd.read_sql_query(
    '''
    SELECT 
        m.anime_id, m.title, r.y_rating_avg, m.year, m.views, r.count AS "количество оценок"
    FROM anime_main m
        LEFT JOIN ratings r ON m.anime_id = r.anime_id
    WHERE r.count > 1000
    ORDER BY r.y_rating_avg DESC
    LIMIT 20

''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\3436406590.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,anime_id,title,y_rating_avg,year,views,количество оценок
0,19627,Re:Zero. Жизнь с нуля в альтернативном мире 4,9.569247,2026,3478087,1899
1,10818,Аватар: Легенда об Аанге,9.457152,2005,1221548,3279
2,23330,Невероятное приключение ДжоДжо: Гонка «Стально...,9.443937,2026,1505309,1534
3,24920,Освободите эту ведьму,9.370480,2026,1226663,2019
4,1509,Вайолет Эвергарден — Фильм,9.286271,2020,227353,2309
5,1486,Унесенные призраками,9.244898,2001,349673,4606
6,481,Ходячий замок,9.196237,2004,528838,4571
7,13329,Ателье колдовских колпаков,9.176184,2026,2764373,1436
8,1512,Ван-Пис,9.172050,1999,9167028,3034
9,15269,Звёздное дитя 3,9.166014,2026,2283954,3319


## Распределение по годам выпуска

In [80]:
df = pd.read_sql_query(
    '''
    SELECT 
        m.year, ROUND(AVG(r.y_rating_avg), 2) AS "средний рейтинг", SUM(m.views) AS "сумма посещений", COUNT(m.anime_id) AS "количество аниме"
    FROM anime_main m
        LEFT JOIN ratings r ON m.anime_id = r.anime_id
    WHERE m.year > 1970 and r.y_rating_avg > 0
    GROUP BY m.year
    ORDER BY m.year DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\3661637132.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,year,средний рейтинг,сумма посещений,количество аниме
0,2026,7.08,122285294,218
1,2025,6.76,168574104,529
2,2024,6.85,79213566,522
3,2023,6.80,64348430,539
4,2022,6.86,40295583,448
5,2021,6.90,37426059,456
6,2020,6.78,28643234,372
7,2019,6.90,30239298,378
8,2018,6.94,28869999,425
9,2017,6.93,24521868,385


## сезонность: какое время года популярнее для релиза

In [81]:
df = pd.read_sql_query(
    '''
    SELECT 
        CASE m.season
            WHEN 1 THEN 'Зима'
            WHEN 2 THEN 'Весна'
            WHEN 3 THEN 'Лето'
            WHEN 4 THEN 'Осень'
        END AS season_name,
        COUNT(m.anime_id) AS "количество аниме",
        ROUND(AVG(r.y_rating_avg), 2) AS avg_rating,
        ROUND(AVG(m.views), 0) AS avg_views,
        SUM(m.views) AS sum_views
    FROM anime_main AS m
        LEFT JOIN ratings AS r ON m.anime_id = r.anime_id
    WHERE m.season IN (1, 2, 3, 4)
        AND m.year > 1970
        AND m.year < 2027
        AND r.y_rating_avg > 0
    GROUP BY m.season

''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\164992675.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,season_name,количество аниме,avg_rating,avg_views,sum_views
0,Зима,2022,6.85,108031.0,218437735
1,Лето,2085,6.82,93401.0,194741096
2,Весна,2201,6.89,94421.0,207821666
3,Осень,2325,6.86,102229.0,237681305


# genres

## Средний рейтинг каждого жанра

In [82]:
df = pd.read_sql_query(
    '''
    SELECT g.title_ru, AVG(r.y_rating_avg) AS avg_score,
        COUNT(DISTINCT m.anime_id) AS anime_count
    FROM anime_main AS m
        JOIN ratings AS r ON m.anime_id = r.anime_id
        JOIN genres AS g ON m.anime_id = g.anime_id
    GROUP BY g.title_ru
    ORDER BY avg_score DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\1948451597.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,title_ru,avg_score,anime_count
0,Антивойна,9.023742,2
1,Охотники за головами,8.449966,5
2,Темные эльфы,8.356660,4
3,Стимпанк,8.307390,12
4,Террористы,8.176195,14
...,...,...,...
99,Боевые искусства,6.048064,636
100,Эротика,6.007729,50
101,Современное фэнтези,5.857315,38
102,Не японское,5.688386,1155


## Самые популярные жанры

### по количеству встречаемости жанров

In [83]:
df = pd.read_sql_query(
    '''
    SELECT g.title_ru, COUNT(g.anime_id) AS anime_count
    FROM genres AS g 
    GROUP BY g.title_ru
    ORDER BY anime_count DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\1077533219.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,title_ru,anime_count
0,Комедия,4433
1,Экшен,4076
2,Фэнтези,3976
3,Приключения,3495
4,Драма,2552
...,...,...
99,Киборги,5
100,Темные эльфы,4
101,Силовые костюмы,4
102,Антивойна,2


### по рейтингу

In [84]:
df = pd.read_sql_query(
    '''
    SELECT g.title_ru, ROUND(AVG(r.y_rating_avg), 2) AS avg_rating
    FROM genres AS g 
        LEFT JOIN ratings AS r ON g.anime_id = r.anime_id
    GROUP BY g.title_ru
    HAVING COUNT(g.anime_id) > 300
    ORDER BY avg_rating DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\3485734668.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,title_ru,avg_rating
0,Сэйнэн,7.03
1,Демоны,7.02
2,Психология,6.99
3,Сверхъестественное,6.93
4,Повседневность,6.89
5,Школьная жизнь,6.89
6,Детектив,6.86
7,Спорт,6.85
8,Суперспособности,6.82
9,Сёнэн,6.82


## Наиболее часто встречаемые сочетания жанров

In [85]:
df = pd.read_sql_query(
    '''
    SELECT g1.title_ru, g2.title_ru, COUNT(*) AS count
    FROM genres AS g1
        INNER JOIN genres AS g2 ON g1.anime_id = g2.anime_id
            AND g1.genre_id < g2.genre_id
    GROUP BY g1.title_ru, g2.title_ru
    ORDER BY count DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\2378835620.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,title_ru,title_ru,count
0,Приключения,Фэнтези,2132
1,Приключения,Экшен,2122
2,Фэнтези,Экшен,2081
3,Комедия,Фэнтези,1521
4,Сёнэн,Комедия,1377
...,...,...,...
2791,Сёдзё-ай,Сёнэн-ай,1
2792,Пилотируемые роботы,Антиутопия,1
2793,Темное фэнтези,Альтернативная история,1
2794,Драконы,Музыка,1


Рейтинг аниме внутри каждого жанра (лучшее в жанре)

In [86]:
df = pd.read_sql_query(
    '''
    WITH genre_rankings AS (
        SELECT 
            g.title_ru AS genre,
            m.title,
            m.year,
            r.y_rating_avg,
            r.count AS votes,
            ROW_NUMBER() OVER (
                PARTITION BY g.title_ru 
                ORDER BY r.y_rating_avg DESC
            ) AS rank_in_genre,
            ROUND(AVG(r.y_rating_avg) OVER (
                PARTITION BY g.title_ru
            ), 2) AS genre_avg_rating
        FROM genres g
            JOIN anime_main m ON g.anime_id = m.anime_id
            JOIN ratings r ON g.anime_id = r.anime_id
        WHERE r.count >= 500
    )
    SELECT * FROM genre_rankings
    WHERE rank_in_genre <= 5
    ORDER BY genre, rank_in_genre
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\3732070111.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,genre,title,year,y_rating_avg,votes,rank_in_genre,genre_avg_rating
0,Альтернативная история,Ходячий замок,2004,9.196237,4571,1,8.63
1,Альтернативная история,Берсерк,1997,9.073744,5831,2,8.63
2,Альтернативная история,Врата Штейна,2011,9.000210,4769,3,8.63
3,Альтернативная история,Атака титанов 3 | Часть 2,2019,8.955301,5593,4,8.63
4,Альтернативная история,Атака титанов,2013,8.901362,7928,5,8.63
...,...,...,...,...,...,...,...
477,Этти,Золотой парень,1995,8.832806,1579,1,7.41
478,Этти,Реинкарнация безработного: История о приключен...,2021,8.680333,6366,2,7.41
479,Этти,Реинкарнация безработного: История о приключен...,2021,8.652670,7978,3,7.41
480,Этти,Богиня благословляет этот прекрасный мир 3,2024,8.536197,3329,4,7.41


# videos

## популярные студии дубляжа 

(по количеству просмотров)

In [87]:
df = pd.read_sql_query(
    '''
    SELECT 
        dubbing_name,
        COUNT(DISTINCT anime_id) AS anime_dubbed,
        COUNT(*) AS total_episodes,
        SUM(views) AS total_views,
        ROUND(AVG(views), 0) AS "просмотров на 1 серию"
    FROM videos
    WHERE dubbing_name IS NOT NULL
    GROUP BY dubbing_name
    ORDER BY total_views DESC
    LIMIT 20
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\2770433501.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,dubbing_name,anime_dubbed,total_episodes,total_views,просмотров на 1 серию
0,Озвучка AniLibria,2134,72453,27027006,373.0
1,Озвучка AniDUB,3874,83267,6674408,80.0
2,Озвучка Dream Cast,555,16498,2615337,159.0
3,Озвучка РуАниме / DEEP,109,3543,2073153,585.0
4,Озвучка AnimeVost,1222,28261,1774921,63.0
5,Озвучка Animedia,712,11681,1648992,141.0
6,Озвучка AniStar,1953,45850,1567630,34.0
7,Озвучка StudioBand,494,10753,1440373,134.0
8,Субтитры Crunchyroll,1119,13799,1253775,91.0
9,Озвучка SHIZA Project,1975,33922,1063352,31.0


## зависимость рейтинга от количества эпизодов

In [88]:
df = pd.read_sql_query(
    '''
    SELECT 
        CASE 
            WHEN m.episodes_count = 1 THEN '1 эпизод (фильм/спэшл)'
            WHEN m.episodes_count BETWEEN 2 AND 12 THEN '2-12 эпизодов'
            WHEN m.episodes_count BETWEEN 13 AND 26 THEN '13-26 эпизодов'
            WHEN m.episodes_count BETWEEN 27 AND 52 THEN '27-52 эпизода'
            WHEN m.episodes_count > 52 THEN '52+ эпизодов'
        END AS episode_bucket,
        COUNT(*) AS anime_count,
        ROUND(AVG(r.y_rating_avg), 2) AS avg_rating,
        ROUND(AVG(m.views), 0) AS avg_views,
        ROUND(AVG(m.comments_count), 0) AS avg_comments,
        ROUND(AVG(m.lists_count), 0) AS avg_in_lists
    FROM anime_main AS m
        LEFT JOIN ratings AS r ON m.anime_id = r.anime_id
    WHERE m.episodes_count > 0
    GROUP BY episode_bucket
    ORDER BY avg_rating DESC
''',
    posgdb.conn
)
df

C:\Users\gusen\AppData\Local\Temp\ipykernel_45568\1773579497.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


,episode_bucket,anime_count,avg_rating,avg_views,avg_comments,avg_in_lists
0,13-26 эпизодов,2080,6.93,130548.0,78.0,2771.0
1,52+ эпизодов,226,6.89,200534.0,109.0,1762.0
2,2-12 эпизодов,3796,6.76,110615.0,70.0,2549.0
3,27-52 эпизода,536,6.52,51722.0,21.0,841.0
4,1 эпизод (фильм/спэшл),3087,6.45,28467.0,9.0,974.0
